In [1]:
# @title Package
from natsort import natsorted
import numpy as np
import seaborn as sns
import pandas as pd

import matplotlib.pyplot as plt
import os

import torch
import torch.nn as nn
import torch.nn.functional as F
import scipy as sp
import scipy.signal as signal
import torchaudio
import math
from sklearn import svm

import torchvision
import torchvision.transforms as transforms
import torchaudio.models as audio_models

from torch.utils.data import DataLoader
from torch.utils.data import TensorDataset

import time

lib_dir = '/content/drive/MyDrive/Project/BrainRegionId/Project44/Code'
os.chdir(lib_dir)
print('library directory: ' + lib_dir)
from modules.networks_clf import *
from modules.signal import spectro_norm, lfp_spectro
from modules.data import *
from modules.metrics import accu_fun

library directory: /content/drive/MyDrive/Project/BrainRegionId/Project44/Code


In [2]:
# @title Load device
dtype = torch.float
# Check whether GPU is available
if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

!nvidia-smi -L


GPU 0: NVIDIA A100-SXM4-40GB (UUID: GPU-089c9f4d-9670-54f5-bd64-d9ebe5597641)


In [3]:
# Set the signal parameters
spectro_args = {
    'nfft':800,
    'power':1,
    'LFP_bound':[0, 500],
    'MUA_bound':[500, 2000],
    'spectro_img':[224, 28],
    'LFP_img':[56 * 4, 28],
    'MUA_img':[0, 28],
    'sampling_lfp':2500,
    'sampling_mua':5000,
    'Log':False,
}

dict_dir = '/content/drive/MyDrive/Project/BrainRegionId/Project37/Data/dat'
acronym_list = acronym_list_gen(dict_dir)

In [4]:
# @title Load data
file_dir = '/content/drive/MyDrive/Project/BrainRegionId/Project43/Data/dat/completed/'
brain_signal_lfp = torch.load('/content/drive/MyDrive/Project/BrainRegionId/Project43/Result/brain_signal_lfp/brain_signal_lfp.pt', weights_only=False)
list_dict = torch.load(file_dir + '/list_dict.pt', weights_only=False)

brain_region_index = list_dict['brain_region_index']
brain_region_index_Cosmos = list_dict['brain_region_index_Cosmos']
coordinate_list = list_dict['coordinate_list']
acronym_selec_list = list_dict['acronym_selec_list']

In [5]:
if len(brain_signal_lfp) == len(brain_region_index):
    print('Matched, no damage!')

Matched, no damage!


In [6]:
community_dict = torch.load('/content/drive/MyDrive/Project/BrainRegionId/Science/results/communities/community_dict.pt', weights_only=False)

In [7]:
def iter_gen(train_ind, valid_ind, test_ind, brain_signal_lfp, brain_region_index0, coordinate_list):
    data_train = TensorDataset(brain_signal_lfp[train_ind,:], brain_region_index0[train_ind], coordinate_list[train_ind])
    data_valid = TensorDataset(brain_signal_lfp[valid_ind,:], brain_region_index0[valid_ind], coordinate_list[valid_ind])
    data_test = TensorDataset(brain_signal_lfp[test_ind,:], brain_region_index0[test_ind], coordinate_list[test_ind])

    train_iter = DataLoader(data_train, batch_size=128, shuffle=True)
    valid_iter = DataLoader(data_valid, batch_size=128, shuffle=True)
    test_iter = DataLoader(data_test, batch_size=128, shuffle=True)

    return train_iter, valid_iter, test_iter

In [8]:
# @title Training function
def net_train_AnyNet_L(train_iter, valid_iter, Classifier, spectro_args, train_args, key, ind, resolution, device):
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(Classifier.parameters(), train_args['lr'])
    # print('lr: ' + train_args['lr'])
    acu_array_train = []
    acu_array_valid = []
    time_array_train = []
    time_array_valid = []
    acu_valid_max = 0
    for epoch in range(0, train_args['epochs']):
        Classifier.train()
        acu_train = []
        epoch0 = 0
        time0_train = time.time()
        for x_train1, y_train, coordinate_train in train_iter:
            x_train = lfp_spectro(x_train1, spectro_args, train_args)
            y_train = y_train.to(device)
            py_train = Classifier(x_train.to(device))
            del x_train, x_train1
            if epoch0 % 800 == 0:
                if epoch0 == 0 and epoch == 0:
                    if accu_fun(py_train, y_train) == (torch.sum(torch.argmax(py_train, dim=1) == y_train) / y_train.size(0)).detach().cpu():
                        print('accu_fun is correct>>>>>>>>>>>>>>>>>>>>>>>>>>>>')
                    else:
                        print('accu_fun is wrong>>>>>>>>>>>>>>>>>>>>>>>>>>>>>>')
                        return
                print(accu_fun(py_train, y_train))
            # print(y_train.size())
            L = loss_fn(py_train,y_train.to(device))
            optimizer.zero_grad()
            L.backward()
            optimizer.step()
            acu_train.append(accu_fun(py_train, y_train))
            epoch0 += 1
        print(f'Acu Train: {torch.mean(torch.tensor(acu_train))}')

        acu_array_train.append(torch.mean(torch.tensor(acu_train)))
        time_array_train.append(time.time() - time0_train)

        Classifier.eval()
        acu_valid = []
        time0_valid = time.time()
        for x_valid1, y_valid, coordinate_valid in valid_iter:
            x_valid = lfp_spectro(x_valid1, spectro_args, train_args)
            y_valid = y_valid.to(device)
            py_valid = Classifier(x_valid.to(device))
            del x_valid, x_valid1
            acu_valid.append(accu_fun(py_valid, y_valid))
            # acu_valid.append((torch.sum(torch.argmax(py_valid, dim=1) == y_valid) / y_valid.size(0)).detach().cpu())

        print(f'Acu valid: {torch.mean(torch.tensor(acu_valid))}')
        if torch.mean(torch.tensor(acu_train)) > train_args['overfitting_thres']:
            if acu_valid_max < torch.mean(torch.tensor(acu_valid)):
                torch.save(Classifier, train_args['save_dir'] + f'/Model/Community/Top/AnyNet_L_AllenCommunity_{key}_{ind}.pth')
                torch.save(epoch, train_args['save_dir'] + f'/Model/Community/Top/AnyNet_L_AllenCommunity_{key}_epoch{ind}.pt')
                acu_valid_max = torch.mean(torch.tensor(acu_valid))

        acu_array_valid.append(torch.mean(torch.tensor(acu_valid)))
        time_array_valid.append(time.time() - time0_valid)

    torch.save(acu_array_train, train_args['save_dir'] + f'/Model/Community/Top/AnyNet_L_AllenCommunity_{key}_train_acu{ind}.pt')
    torch.save(acu_array_valid, train_args['save_dir'] + f'/Model/Community/Top/AnyNet_L_AllenCommunity_{key}_valid_acu{ind}.pt')

    return


def net_train_ViT_L(train_iter, valid_iter, Classifier, spectro_args, train_args, key, ind, resolution, device):
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(Classifier.parameters(), train_args['lr'])
    # print('lr: ' + train_args['lr'])
    acu_array_train = []
    acu_array_valid = []
    time_array_train = []
    time_array_valid = []
    acu_valid_max = 0
    for epoch in range(0, train_args['epochs']):
        Classifier.train()
        acu_train = []
        epoch0 = 0
        time0_train = time.time()
        for x_train1, y_train, coordinate_train in train_iter:
            x_train = lfp_spectro(x_train1, spectro_args, train_args)
            y_train = y_train.to(device)
            py_train = Classifier(x_train.to(device))
            del x_train, x_train1
            if epoch0 % 800 == 0:
                print((torch.sum(torch.argmax(py_train, dim=1) == y_train) / y_train.size(0)).detach().cpu())
            L = loss_fn(py_train,y_train.to(device))
            optimizer.zero_grad()
            L.backward()
            optimizer.step()
            acu_train.append((torch.sum(torch.argmax(py_train, dim=1) == y_train) / y_train.size(0)).detach().cpu())
            epoch0 += 1
        print(f'Acu Train: {torch.mean(torch.tensor(acu_train))}')

        acu_array_train.append(torch.mean(torch.tensor(acu_train)))
        time_array_train.append(time.time() - time0_train)

        Classifier.eval()
        acu_valid = []
        time0_valid = time.time()
        for x_valid1, y_valid, coordinate_valid in valid_iter:
            x_valid = lfp_spectro(x_valid1, spectro_args, train_args)
            y_valid = y_valid.to(device)
            py_valid = Classifier(x_valid.to(device))
            del x_valid, x_valid1
            acu_valid.append((torch.sum(torch.argmax(py_valid, dim=1) == y_valid) / y_valid.size(0)).detach().cpu())

        print(f'Acu valid: {torch.mean(torch.tensor(acu_valid))}')
        if torch.mean(torch.tensor(acu_train)) > train_args['overfitting_thres']:
            if acu_valid_max < torch.mean(torch.tensor(acu_valid)):
                torch.save(Classifier, train_args['save_dir'] + f'/Model/Community/Top/ViT_L_AllenCommunity_{key}_{ind}.pth')
                torch.save(epoch, train_args['save_dir'] + f'/Model/Community/Top/ViT_L_AllenCommunity_{key}_epoch{ind}.pt')
                acu_valid_max = torch.mean(torch.tensor(acu_valid))

        acu_array_valid.append(torch.mean(torch.tensor(acu_valid)))
        time_array_valid.append(time.time() - time0_valid)

    torch.save(torch.tensor(acu_array_train), train_args['save_dir'] + f'/Model/Community/Top/ViT_L_AllenCommunity_{key}_train_acu{ind}.pt')
    torch.save(torch.tensor(acu_array_valid), train_args['save_dir'] + f'/Model/Community/Top/ViT_L_AllenCommunity_{key}_valid_acu{ind}.pt')

    return

def net_train_RNN_L(train_iter, valid_iter, Classifier, spectro_args, train_args, key, ind, resolution, device):
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(Classifier.parameters(), train_args['lr'])
    # print('lr: ' + train_args['lr'])
    acu_array_train = []
    acu_array_valid = []
    time_array_train = []
    time_array_valid = []
    acu_valid_max = 0
    for epoch in range(0, train_args['epochs']):
        Classifier.train()
        acu_train = []
        time0_train = time.time()
        epoch0 = 0
        for x_train1, y_train, coordinate_train in train_iter:
            x_train = lfp_spectro(x_train1, spectro_args, train_args).squeeze(1).permute(0, 2, 1)
            y_train = y_train.to(device)
            py_train = Classifier(x_train.to(device))
            del x_train, x_train1
            if epoch0 % 800 == 0:
                print((torch.sum(torch.argmax(py_train, dim=1) == y_train) / y_train.size(0)).detach().cpu())
            L = loss_fn(py_train,y_train.to(device))
            optimizer.zero_grad()
            L.backward()
            optimizer.step()
            acu_train.append((torch.sum(torch.argmax(py_train, dim=1) == y_train) / y_train.size(0)).detach().cpu())
            epoch0 += 1
        print(f'Acu Train: {torch.mean(torch.tensor(acu_train))}')

        acu_array_train.append(torch.mean(torch.tensor(acu_train)))
        time_array_train.append(time.time() - time0_train)

        Classifier.eval()
        acu_valid = []
        time0_valid = time.time()
        for x_valid1, y_valid, coordinate_valid in valid_iter:
            x_valid = lfp_spectro(x_valid1, spectro_args, train_args).squeeze(1).permute(0, 2, 1)
            y_valid = y_valid.to(device)
            py_valid = Classifier(x_valid.to(device))
            del x_valid, x_valid1
            acu_valid.append((torch.sum(torch.argmax(py_valid, dim=1) == y_valid) / y_valid.size(0)).detach().cpu())

        print(f'Acu valid: {torch.mean(torch.tensor(acu_valid))}')
        if torch.mean(torch.tensor(acu_train)) > train_args['overfitting_thres']:
            if acu_valid_max < torch.mean(torch.tensor(acu_valid)):
                torch.save(Classifier, train_args['save_dir'] + f'/Model/Community/Top/RNN_L_AllenCommunity_{key}_{ind}.pth')
                torch.save(epoch, train_args['save_dir'] + f'/Model/Community/Top/RNN_L_AllenCommunity_{key}_epoch{ind}.pt')
                acu_valid_max = torch.mean(torch.tensor(acu_valid))


        acu_array_valid.append(torch.mean(torch.tensor(acu_valid)))
        time_array_valid.append(time.time() - time0_valid)

    torch.save(torch.tensor(acu_array_train), train_args['save_dir'] + f'/Model/Community/Top/RNN_L_AllenCommunity_{key}_train_acu{ind}.pt')
    torch.save(torch.tensor(acu_array_valid), train_args['save_dir'] + f'/Model/Community/Top/RNN_L_AllenCommunity_{key}_valid_acu{ind}.pt')

    return


In [9]:
communities_accu_indiv = torch.load('/content/drive/MyDrive/Project/BrainRegionId/Science/results/communities/communities_accu_indiv.pt', weights_only=False)

In [10]:
marked_community = []
for community in np.unique(community_dict[1.0]['communities_label']):
    df_indiv0 = communities_accu_indiv[1.0][communities_accu_indiv[1.0]['acronym_test'] == community]
    for classifier_name in ['AnyNet', 'ViT', 'RNN']:
        if df_indiv0[df_indiv0['model_type'] == classifier_name]['acu_test'].mean() > 0.60:
            marked_community.append(community)

marked_community = np.unique(np.array(marked_community))

In [11]:
marked_community

array([ 0,  1,  2,  9, 10, 11, 16, 19, 22, 28, 29])

In [12]:
community_dict[1.0]['communities_label']

array([ 0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
        0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
        0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
        1,  1,  1,  1,  1,  1,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,
        2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,
        2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  2,  3,  3,
        3,  3,  3,  3,  3,  3,  3,  3,  3,  3,  3,  3,  3,  3,  3,  4,  4,
        4,  4,  4,  4,  4,  4,  4,  4,  4,  4,  4,  4,  5,  5,  5,  6,  6,
        6,  6,  6,  6,  6,  6,  6,  6,  6,  6,  7,  7,  7,  8,  8,  8,  9,
        9,  9,  9,  9,  9,  9,  9,  9,  9,  9,  9,  9, 10, 10, 10, 10, 10,
       10, 10, 10, 10, 10, 10, 11, 11, 11, 12, 12, 12, 12, 12, 12, 12, 12,
       12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12, 12,
       12, 12, 13, 13, 13, 13, 13, 13, 13, 13, 13, 13, 14, 14, 14, 14, 14,
       14, 14, 14, 14, 14

In [13]:
np.argwhere(marked_community == 1).flatten()

array([1])

In [14]:
len(np.unique(marked_community))

11

In [35]:
acronym_communities_accu_div = torch.load('/content/drive/MyDrive/Project/BrainRegionId/Science/results/communities/acronym_communities_accu_indiv.pt', weights_only=False)
selec_br = []
ig_br = []
for model in ['AnyNet', 'ViT', 'RNN']:
    dict_model = acronym_communities_accu_div[1.0][acronym_communities_accu_div[1.0]['model_type'] == model]
    for acronym in acronym_list:
        if dict_model[dict_model['acronym_test'] == acronym]['acu_test'].mean() < 0.5:
            ig_br.append(acronym)

ig_br = np.unique(np.array(ig_br))
selec_br = np.setdiff1d(np.array(acronym_list), ig_br)

In [41]:
'DG-sg' in selec_br

True

In [42]:
len(selec_br)

307

In [45]:
len(torch.argwhere(comm_ind_top == comm_ii).flatten())

1280

In [ ]:
save_dir = '/content/drive/MyDrive/Project/BrainRegionId/Science/'

for resolution in [1.0]:
    communities = community_dict[resolution]['communities']
    communities_label = community_dict[resolution]['communities_label']
    communities_acronym = community_dict[resolution]['communities_acronym']

    id_community_mapping = []
    for acronym in acronym_list:
        if acronym in communities_acronym:
            if communities_label[np.argwhere(communities_acronym == acronym).flatten()] in marked_community:
                if acronym in selec_br:
                    label_top = np.argwhere(marked_community == communities_label[np.argwhere(communities_acronym == acronym).flatten()]).flatten()
                    id_community_mapping.append(label_top)
                else:
                    id_community_mapping.append(np.array([-1]))
            else:
                id_community_mapping.append(np.array([-1]))

        else:
            id_community_mapping.append(np.array([-1]))

    id_community_mapping = np.array(id_community_mapping)

    brain_region_index_intersect = id_community_mapping[brain_region_index].squeeze(-1)
    dropout_index = np.argwhere(brain_region_index_intersect == np.array([-1]))[:, 0]

    brain_region_index_intersect = torch.tensor(brain_region_index_intersect, dtype=torch.long)

    key = 'stimOff_times'
    model_dir = '/content/drive/MyDrive/Project/BrainRegionId/Project44/Model/Allen'
    subject_od_ind = torch.load(model_dir + f'/subject_od_ind_Allen_{key}{0}.pt', weights_only=False)
    subject_od_list = torch.load(model_dir + f'/subject_od_list_Allen_{key}{0}.pt', weights_only=False)
    train_ind, valid_ind, test_ind, test_subject_ind = dat_ind_gen(list_dict, subject_od_ind, key)
    train_ind = np.setdiff1d(train_ind, dropout_index)
    valid_ind = np.setdiff1d(valid_ind, dropout_index)
    test_ind = np.setdiff1d(test_ind, dropout_index)

    ind_selec = []
    for comm_ii, comm in enumerate(marked_community):
        comm_ind_top = brain_region_index_intersect
        ind_top_len = len(torch.argwhere(comm_ind_top == comm_ii).flatten())

        ind_selec.append(torch.argwhere(comm_ind_top == comm_ii).flatten()[np.random.choice(ind_top_len, 1000, replace=False)])

    ind_selec = torch.concat(ind_selec).numpy()

    # train_ind = np.intersect1d(train_ind, ind_selec)
    valid_ind = np.intersect1d(valid_ind, ind_selec)
    test_ind = np.intersect1d(test_ind, ind_selec)
    print(len(train_ind))
    print(len(valid_ind))
    print(len(test_ind))

    train_iter_AnyNet, valid_iter_AnyNet, test_iter_AnyNet = iter_gen(train_ind, valid_ind, test_ind, brain_signal_lfp, brain_region_index_intersect, coordinate_list)

    c0 = 64 * 4
    k0 = 1.0

    model_args = {
        'arch':((2,c0 * 2,1,k0), (2,c0 * 3,1,k0), (2,c0 * 4,1,k0), (2,c0 * 5,1,k0)),
        'stem_channels':c0,
    }
    train_args = {
        'overfitting_thres':0.60,
        'lr':5e-4,
        'norm':True,
        'temp':[True, True],
        'epochs':10,
        'save_dir':save_dir,
    }


    for ind in range(0, 1):
        Classifier = AnyNet_L(model_args['arch'], model_args['stem_channels'], frequency_bin=spectro_args['spectro_img'][0], num_classes=len(np.unique(marked_community))).to(device)
        Classifier.apply(init_cnn)
        net_train_AnyNet_L(train_iter_AnyNet, valid_iter_AnyNet, Classifier.to(device), spectro_args, train_args, key, ind, resolution, device)

    train_iter_ViT, valid_iter_ViT, test_iter_ViT = iter_gen(train_ind, valid_ind, test_ind, brain_signal_lfp, brain_region_index_intersect, coordinate_list)

    train_args['epochs'] = 50
    for ind in range(0, 1):
        img_size, patch_size = (224, 28), (28, 28)
        num_hiddens, mlp_num_hiddens, num_heads, num_blks = 512, 2048, 8, 4
        emb_dropout, blk_dropout = 0.1, 0.1
        Classifier = ViT_L(spectro_args['spectro_img'][0], img_size, patch_size, num_hiddens, mlp_num_hiddens, num_heads, num_blks, emb_dropout, blk_dropout, num_classes=len(np.unique(marked_community))).to(device)
        net_train_ViT_L(train_iter_ViT, valid_iter_ViT, Classifier.to(device), spectro_args, train_args, key, ind, resolution, device)

    del train_iter_ViT, valid_iter_ViT, test_iter_ViT
    train_iter_RNN, valid_iter_RNN, test_iter_RNN = iter_gen(train_ind, valid_ind, test_ind, brain_signal_lfp, brain_region_index_intersect, coordinate_list)
    train_args['epochs'] = 30

    RNN_args = {
    'input_size':224,
    'hidden_size':512 * 2,
    'num_layers':2,
    'category_num':len(np.unique(marked_community)),
    'data_len':28,
    }
    for ind in range(0, 1):
        Classifier = RNN_L(spectro_args['spectro_img'][0], RNN_args).to(device)
        net_train_RNN_L(train_iter_RNN, valid_iter_RNN, Classifier.to(device), spectro_args, train_args, key, ind, resolution, device)


513097
664
687
accu_fun is correct>>>>>>>>>>>>>>>>>>>>>>>>>>>>
tensor(0.2344)
tensor(0.6641)
tensor(0.7656)
tensor(0.7109)
tensor(0.7734)
tensor(0.8047)
Acu Train: 0.7263772487640381
Acu valid: 0.65234375
tensor(0.8203)
tensor(0.8438)
tensor(0.8516)
tensor(0.8672)
tensor(0.8438)
tensor(0.7891)
Acu Train: 0.8246648907661438
Acu valid: 0.7083333134651184
tensor(0.8984)
tensor(0.8047)
tensor(0.8828)
tensor(0.8984)
tensor(0.8906)
tensor(0.8828)
Acu Train: 0.8707770109176636
Acu valid: 0.6805555820465088
tensor(0.9062)
tensor(0.8984)
tensor(0.8906)
tensor(0.9297)
tensor(0.8984)
tensor(0.9375)
Acu Train: 0.9020134210586548
Acu valid: 0.75390625
tensor(0.9297)
tensor(0.9375)
tensor(0.9297)
tensor(0.9453)
tensor(0.9141)
tensor(0.9609)
Acu Train: 0.9241657257080078
Acu valid: 0.7309028506278992
tensor(0.9453)
tensor(0.9531)
tensor(0.9297)
tensor(0.9531)
tensor(0.9531)
tensor(0.8750)
Acu Train: 0.9406057596206665
Acu valid: 0.6901041865348816
tensor(0.9609)
tensor(0.9375)
tensor(0.9531)
tensor(0

In [ ]:
# save_dir = '/content/drive/MyDrive/Project/BrainRegionId/Science/'

# for resolution in [1.0]:
#     communities = community_dict[resolution]['communities']
#     communities_label = community_dict[resolution]['communities_label']
#     communities_acronym = community_dict[resolution]['communities_acronym']

#     id_community_mapping = []
#     for acronym in acronym_list:
#         if acronym in communities_acronym:
#             if communities_label[np.argwhere(communities_acronym == acronym).flatten()] in marked_community:
#                 label_top = np.argwhere(marked_community == communities_label[np.argwhere(communities_acronym == acronym).flatten()]).flatten()
#                 id_community_mapping.append(label_top)
#             id_community_mapping.append(np.array([-1]))

#         else:
#             id_community_mapping.append(np.array([-1]))

#     id_community_mapping = np.array(id_community_mapping)

#     brain_region_index_intersect = id_community_mapping[brain_region_index].squeeze(-1)
#     dropout_index = np.argwhere(brain_region_index_intersect == np.array([-1]))[:, 0]

#     brain_region_index_intersect = torch.tensor(brain_region_index_intersect, dtype=torch.long)

#     key = 'stimOff_times'
#     model_dir = '/content/drive/MyDrive/Project/BrainRegionId/Project44/Model/Allen'
#     subject_od_ind = torch.load(model_dir + f'/subject_od_ind_Allen_{key}{0}.pt', weights_only=False)
#     subject_od_list = torch.load(model_dir + f'/subject_od_list_Allen_{key}{0}.pt', weights_only=False)
#     train_ind, valid_ind, test_ind, test_subject_ind = dat_ind_gen(list_dict, subject_od_ind, key)
#     train_ind = np.setdiff1d(train_ind, dropout_index)
#     valid_ind = np.setdiff1d(valid_ind, dropout_index)
#     test_ind = np.setdiff1d(test_ind, dropout_index)
#     train_iter_AnyNet, valid_iter_AnyNet, test_iter_AnyNet = iter_gen(train_ind, valid_ind, test_ind, brain_signal_lfp, brain_region_index_intersect, coordinate_list)

#     c0 = 64 * 4
#     k0 = 1.0

#     model_args = {
#         'arch':((2,c0 * 2,1,k0), (2,c0 * 3,1,k0), (2,c0 * 4,1,k0), (2,c0 * 5,1,k0)),
#         'stem_channels':c0,
#     }
#     train_args = {
#         'overfitting_thres':0.60,
#         'lr':5e-4,
#         'norm':True,
#         'temp':[True, True],
#         'epochs':10,
#         'save_dir':save_dir,
#     }


#     for ind in range(0, 1):
#         Classifier = AnyNet_L(model_args['arch'], model_args['stem_channels'], frequency_bin=spectro_args['spectro_img'][0], num_classes=len(np.unique(marked_community))).to(device)
#         Classifier.apply(init_cnn)
#         net_train_AnyNet_L(train_iter_AnyNet, valid_iter_AnyNet, Classifier.to(device), spectro_args, train_args, key, ind, resolution, device)

#     train_iter_ViT, valid_iter_ViT, test_iter_ViT = iter_gen(train_ind, valid_ind, test_ind, brain_signal_lfp, brain_region_index_intersect, coordinate_list)

#     train_args['epochs'] = 50
#     for ind in range(0, 1):
#         img_size, patch_size = (224, 28), (28, 28)
#         num_hiddens, mlp_num_hiddens, num_heads, num_blks = 512, 2048, 8, 4
#         emb_dropout, blk_dropout = 0.1, 0.1
#         Classifier = ViT_L(spectro_args['spectro_img'][0], img_size, patch_size, num_hiddens, mlp_num_hiddens, num_heads, num_blks, emb_dropout, blk_dropout, num_classes=len(np.unique(marked_community))).to(device)
#         net_train_ViT_L(train_iter_ViT, valid_iter_ViT, Classifier.to(device), spectro_args, train_args, key, ind, resolution, device)

#     del train_iter_ViT, valid_iter_ViT, test_iter_ViT
#     train_iter_RNN, valid_iter_RNN, test_iter_RNN = iter_gen(train_ind, valid_ind, test_ind, brain_signal_lfp, brain_region_index_intersect, coordinate_list)
#     train_args['epochs'] = 30

#     RNN_args = {
#     'input_size':224,
#     'hidden_size':512 * 2,
#     'num_layers':2,
#     'category_num':len(np.unique(marked_community)),
#     'data_len':28,
#     }
#     for ind in range(0, 1):
#         Classifier = RNN_L(spectro_args['spectro_img'][0], RNN_args).to(device)
#         net_train_RNN_L(train_iter_RNN, valid_iter_RNN, Classifier.to(device), spectro_args, train_args, key, ind, resolution, device)


In [ ]:
from google.colab import runtime
runtime.unassign()